# Week 3 – Feature Engineering on the Updated Retail Dataset

## Project
**Demand Forecasting for Retail Inventory Optimization**

## Objective

In this section, we explore four text vectorization methods:

1. **Bag of Words (BoW)**
2. **TF-IDF**
3. **Word2Vec**
4. **BERT / Transformer Embeddings**

These methods convert text-like fields into numerical vectors that can later be combined with structured retail features for machine learning models.

## Important Note

The updated retail dataset is primarily a **structured tabular dataset**.  
The text-like fields in this dataset are mostly **short categorical/business labels** such as:

- `Category`
- `Store_Type`
- `Festival_Name`
- `Festival_Type`
- `Product_Name`
- `Sub_Category`
- `Brand_Name`
- `Brand_Tier`
- `Promotion_Type`

Therefore, these vectorization methods are being explored as part of the project requirement, but they are **not the primary feature engineering strategy** for this dataset.  
Later, we will also apply more appropriate tabular feature engineering methods.

## Target Variable

The target variable remains:

`Daily_Units_Sold`

In [1]:
import numpy as np
import pandas as pd
import scipy
import sklearn
import gensim

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("gensim:", gensim.__version__)

numpy: 1.26.4
pandas: 2.2.2
scipy: 1.13.1
sklearn: 1.6.1
gensim: 4.3.3


In [2]:
# @title Import basic libraries for data handling
#import pandas as pd
#import numpy as np

# Import libraries for text processing
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Import libraries for Word2Vec
from gensim.models import Word2Vec

# Import library for contextual embeddings
!pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer

# Display options for readability
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


## Load the Updated Dataset

We first load the updated retail dataset and inspect its shape and column names.

The updated dataset is expected to contain:
- 2,000 rows
- 58 columns

In [3]:
# @title Load the updated dataset
df = pd.read_csv("/content/master_retail_dataset_v2.csv")

# Display shape of the dataset
print("Dataset shape:", df.shape)

# Preview first 5 rows
df.head()

Dataset shape: (2000, 58)


,Date,DayOfWeek,Month,Year,Is_Weekend,Is_Holiday,Product_ID,Category,Base_Price,Discount_Percentage,Current_Price,Store_ID,Store_Type,Competitor_Price,Footfall_Index,Avg_Temperature,Rainfall_mm,Social_Media_Sentiment,Lead_Time_Days,Safety_Stock_Level,Stock_On_Hand,Daily_Units_Sold,Festival_Name,Festival_Type,UPC_EAN,Product_Name,Sub_Category,Brand_Name,Brand_Tier,Shelf_Life_Days,No_of_Checkout_Counters,Avg_Billing_Time_min,Local_Population_Density,Product_Age_Days,No_of_Customer_Purchases,Unit_Cost,Promotional_Campaign_Flag,Competitor_Promotion_Flag,Google_Trends_Current_Wk,Google_Trends_Lag_1w,Shelf_Capacity,Promotion_Type,Marketing_Spend,Payday_Flag,School_Vacation_Flag,Local_Event_Flag,Launch_Date,Seasonal_Product_Flag,Online_Sales_Units,In_Store_Sales_Units,App_Traffic_Index,Website_Visits,Backorder_Flag,Supplier_Delay_Days,Fill_Rate_Pct,Loyalty_Program_Usage_Count,Repeat_Purchase_Rate,Avg_Basket_Size
0,04/13/2023,3,4,2023,0,0,1002,Grocery,410.09,0.01,405.99,15,Urban,411.58,127.6,29.1,8.5,0.63,7,141,471,83,NaN,NaN,8148570840155,Nestle Beverages 1002,Beverages,Nestle,Premium,54,6,5.9,12371,228,22,322.20,N,N,32,32,30,NaN,1247,0,0,0,08/28/2022,Y,12,71,488,896,N,0,88.93,11,0.14,3.77
1,12/15/2023,4,12,2023,0,0,1014,Home,89.98,0.30,62.99,7,Rural,90.37,56.9,1.9,5.3,0.35,11,59,263,102,NaN,NaN,7203826864190,Tupperware Decor 1014,Decor,Tupperware,Budget,2244,11,5.2,1046,1822,69,47.96,Y,Y,77,32,99,Flash Sale,1942,0,1,0,12/19/2018,Y,31,71,619,1406,N,2,90.16,31,0.14,1.48
2,09/28/2023,3,9,2023,0,0,1015,Electronics,404.88,0.14,348.20,17,Urban,408.07,191.8,14.5,2.8,0.77,4,111,302,89,NaN,NaN,5094762337870,Sony Computers 1015,Computers,Sony,Mid,1196,11,7.4,9278,1759,83,319.44,N,Y,19,77,71,NaN,1042,0,0,0,12/04/2018,N,32,57,626,1432,N,6,88.00,23,0.38,1.07
3,04/17/2023,0,4,2023,0,0,1012,Grocery,347.97,0.43,198.34,12,Urban,349.20,71.6,24.4,1.3,0.86,11,38,389,98,NaN,NaN,16104203210,Nestle Spices 1012,Spices,Nestle,Premium,7,8,4.4,7650,1330,21,266.76,Y,N,33,19,28,Flash Sale,4105,0,0,0,08/26/2019,N,32,66,626,1432,N,2,91.17,7,0.13,4.67
4,03/13/2023,0,3,2023,0,0,1016,Grocery,280.16,0.14,240.94,18,Urban,252.64,185.5,26.0,0.4,0.36,14,46,262,100,NaN,NaN,8572879201082,Tata Spices 1016,Spices,Tata,Premium,10,13,2.6,5779,1127,46,172.32,N,Y,60,33,95,NaN,706,0,0,0,02/10/2020,N,23,77,564,1191,N,0,89.39,9,0.16,2.17


## Identify Text-Like Features

The dataset is mostly structured, but some fields contain business labels that can be treated as text-like features for vectorization.

The selected columns are:

- `Category`
- `Store_Type`
- `Festival_Name`
- `Festival_Type`
- `Product_Name`
- `Sub_Category`
- `Brand_Name`
- `Brand_Tier`
- `Promotion_Type`

In [4]:
# @title List of text-like columns selected for vectorization
text_columns = [
    "Category",
    "Store_Type",
    "Festival_Name",
    "Festival_Type",
    "Product_Name",
    "Sub_Category",
    "Brand_Name",
    "Brand_Tier",
    "Promotion_Type"
]

# Show number of unique values in each selected text column
for col in text_columns:
    print(f"{col}: {df[col].nunique()} unique values")

Category: 5 unique values
Store_Type: 2 unique values
Festival_Name: 8 unique values
Festival_Type: 3 unique values
Product_Name: 100 unique values
Sub_Category: 20 unique values
Brand_Name: 19 unique values
Brand_Tier: 3 unique values
Promotion_Type: 4 unique values


## Create a Combined Text Field

Most vectorization methods work on a single text document per row.

To apply these methods, we combine the selected text-like columns into one consolidated text field called:

`combined_text`

This creates a simple text representation of each retail record.

In [5]:
# Fill missing values with empty string just in case
df[text_columns] = df[text_columns].fillna("")

# Combine selected columns into one text string per row
df["combined_text"] = df[text_columns].astype(str).agg(" ".join, axis=1)

# Preview the combined text
df[["combined_text"]].head()

,combined_text
0,Grocery Urban Nestle Beverages 1002 Beverages Nestle Premium
1,Home Rural Tupperware Decor 1014 Decor Tupperware Budget Flash Sale
2,Electronics Urban Sony Computers 1015 Computers Sony Mid
3,Grocery Urban Nestle Spices 1012 Spices Nestle Premium Flash Sale
4,Grocery Urban Tata Spices 1016 Spices Tata Premium


## Basic Text Cleaning

Before vectorization, we perform light text cleaning:

- convert text to lowercase
- remove extra spaces
- keep only letters, numbers, and spaces

This is a simple preprocessing step suitable for the current dataset.

In [6]:
# Function to clean text
def clean_text(text):
    """
    Clean input text by:
    1. converting to lowercase
    2. removing special characters
    3. removing extra spaces
    """
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Apply cleaning
df["clean_text"] = df["combined_text"].apply(clean_text)

# Preview cleaned text
df[["combined_text", "clean_text"]].head()

,combined_text,clean_text
0,Grocery Urban Nestle Beverages 1002 Beverages Nestle Premium,grocery urban nestle beverages 1002 beverages nestle premium
1,Home Rural Tupperware Decor 1014 Decor Tupperware Budget Flash Sale,home rural tupperware decor 1014 decor tupperware budget flash sale
2,Electronics Urban Sony Computers 1015 Computers Sony Mid,electronics urban sony computers 1015 computers sony mid
3,Grocery Urban Nestle Spices 1012 Spices Nestle Premium Flash Sale,grocery urban nestle spices 1012 spices nestle premium flash sale
4,Grocery Urban Tata Spices 1016 Spices Tata Premium,grocery urban tata spices 1016 spices tata premium


# 1. Bag of Words (BoW)

## Concept

Bag of Words converts each document into a vector of **word counts**.

If a word appears more often in a row's text, its value in the vector is higher.

### Formula

For document $d_i$ and term $w_j$:

$
x_{ij} = \text{count}(w_j, d_i)
$

where:
- $x_{ij}$ = count of word $w_j$ in document $d_i$

## Why We Are Using It

BoW is:
- simple
- easy to interpret
- useful as a baseline vectorization method

## Limitation

It does **not** capture:
- semantic meaning
- word order
- context

In [7]:
# Create a CountVectorizer object
# max_features limits the vocabulary size so the matrix remains manageable
bow_vectorizer = CountVectorizer(max_features=100)

# Learn vocabulary and transform text into count vectors
X_bow = bow_vectorizer.fit_transform(df["clean_text"])

# Convert sparse matrix to DataFrame for inspection
bow_df = pd.DataFrame(
    X_bow.toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)

print("BoW matrix shape:", bow_df.shape)
bow_df.head()

BoW matrix shape: (2000, 95)


,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,accessories,activewear,allen,amul,ashirvad,audio,beauty,bedding,beverages,bogo,bose,budget,christmas,clothing,computers,dairy,day,decor,dhanteras,discount,diwali,dove,dussehra,eid,electronics,ethnic,fabindia,festive,fitr,flash,formalwear,fragrance,furniture,gifting,grains,grocery,haircare,holi,home,ikea,independence,kitchenware,lakme,levis,loyalty,makeup,mega,mid,mobiles,nestle,new,nivea,only,oreal,philips,premium,rural,sale,samsung,season,seasonal,skincare,sleepwell,solly,sony,spices,tata,tupperware,ul,urban,wear,wedding,winter,year,zara
0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,2,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,2,2,0,0,1,0,0,0,0,0


## Observation

Bag of Words produced a **2000 × 95** sparse feature matrix. Although the vectorizer was configured with `max_features=100`, the effective vocabulary extracted from the cleaned combined text was **95 tokens**.

The generated vocabulary is dominated by:
- short business labels such as categories, brand tiers, and promotion words
- product/store identifiers such as `1002`, `1014`, and similar numeric tokens
- repeated business terms such as product or sub-category names

This confirms that BoW is mainly capturing **token presence and repetition** in structured label-style text rather than rich natural language context. For this dataset, it works as a simple baseline representation, but it mostly reflects category, brand, promotion, and identifier information.


## Note on Combined Text Representation

In the Bag of Words approach, all selected text-like columns were combined into a single document per row.

This is a standard NLP baseline approach, but it may mix semantically different fields such as:
- product
- brand
- store type
- promotion
- festival

This limitation is examined further in the next section.

# 2. TF-IDF

## Concept

TF-IDF stands for **Term Frequency – Inverse Document Frequency**.

It improves upon Bag of Words by giving:
- higher weight to words that are important in a specific document
- lower weight to very common words across many documents

### Formula

$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$

where:

$
\text{IDF}(t) = \log\left(\frac{N}{df(t)}\right)
$

- $N$ = total number of documents
- $df(t)$ = number of documents containing term $t$

## Why We Are Using It

TF-IDF is useful because it highlights **informative terms** and reduces the impact of overly common words.

## Limitation

It still does not capture semantic meaning or context.

In [8]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=100)

# Fit and transform the cleaned text
X_tfidf = tfidf_vectorizer.fit_transform(df["clean_text"])

# Convert sparse matrix to DataFrame for inspection
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

print("TF-IDF matrix shape:", tfidf_df.shape)
tfidf_df.head()

TF-IDF matrix shape: (2000, 95)


,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,accessories,activewear,allen,amul,ashirvad,audio,beauty,bedding,beverages,bogo,bose,budget,christmas,clothing,computers,dairy,day,decor,dhanteras,discount,diwali,dove,dussehra,eid,electronics,ethnic,fabindia,festive,fitr,flash,formalwear,fragrance,furniture,gifting,grains,grocery,haircare,holi,home,ikea,independence,kitchenware,lakme,levis,loyalty,makeup,mega,mid,mobiles,nestle,new,nivea,only,oreal,philips,premium,rural,sale,samsung,season,seasonal,skincare,sleepwell,solly,sony,spices,tata,tupperware,ul,urban,wear,wedding,winter,year,zara
0,0.0,0.0,0.327848,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.641556,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.205741,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.628539,0.0,0.0,0.0,0.0,0.0,0.168795,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.122632,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.301024,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.165315,0.0,0.0,0.000000,0.0,0.0,0.640908,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.208649,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.204371,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.15071,0.208649,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.565457,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.339328,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.585307,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.231802,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.185165,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.661084,0.000000,0.000000,0.000000,0.0,0.131229,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.316718,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.222987,0.0,0.0,0.0,0.0,0.0,0.208258,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.636228,0.0,0.0,0.0,0.0,0.0,0.170860,0.00000,0.222987,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.554513,0.000000,0.000000,0.0,0.124132,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.338691,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.220248,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.180697,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.586439,0.665569,0.000000,0.0,0.131279,0.0,0.0,0.0,0.0,0.0


## Observation

TF-IDF produced a **2000 × 95** weighted feature matrix, which matches the effective vocabulary size observed in the Bag of Words representation.

Compared with BoW, TF-IDF assigns higher weight to terms that are more specific to individual records and lower weight to terms that appear very frequently across many rows. In this dataset, the weighting is especially useful because the combined text contains repeated business labels such as category, brand, and promotion terms.

However, the output also shows that many important tokens are still short structured labels or numeric identifiers rather than descriptive sentences. Therefore, TF-IDF is more informative than BoW for highlighting distinguishing labels, but it is still limited by the short, categorical nature of the text fields.


## Limitation of Combined Text TF-IDF

The previous TF-IDF approach treated all text-like columns as one combined document per row.

However, this dataset contains multiple text-like fields with different meanings, such as:

- `Category`
- `Store_Type`
- `Festival_Name`
- `Festival_Type`
- `Product_Name`
- `Sub_Category`
- `Brand_Name`
- `Brand_Tier`
- `Promotion_Type`

Combining them into a single text string mixes unrelated semantic categories. As a result, the TF-IDF representation may lose interpretability because words from different business domains are treated as if they belong to the same context.

## Improved Approach: Column-wise TF-IDF Vectorization

To preserve the semantic meaning of each text-like feature, TF-IDF can be applied separately to each column rather than combining all columns into one document.

This approach helps to:

- preserve feature-specific context
- reduce semantic mixing
- improve interpretability

In this method, each text column is vectorized independently, and the resulting feature matrices are then concatenated.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create an empty list to store TF-IDF DataFrames for each column
tfidf_features = []

# Apply TF-IDF separately to each text column
for col in text_columns:
    # Initialize TF-IDF vectorizer
    # max_features is limited to keep the feature space manageable
    vectorizer = TfidfVectorizer(max_features=20)

    # Convert the current column to string
    text_data = df[col].astype(str)

    # Fit and transform the column
    col_matrix = vectorizer.fit_transform(text_data)

    # Convert sparse matrix to DataFrame
    col_df = pd.DataFrame(
        col_matrix.toarray(),
        columns=[f"{col}_{word}" for word in vectorizer.get_feature_names_out()]
    )

    # Store the result
    tfidf_features.append(col_df)

# Combine all column-wise TF-IDF features into one DataFrame
tfidf_combined_df = pd.concat(tfidf_features, axis=1)

# Display shape and preview
print("Column-wise TF-IDF shape:", tfidf_combined_df.shape)
tfidf_combined_df.head()

Column-wise TF-IDF shape: (2000, 94)


,Category_beauty,Category_clothing,Category_electronics,Category_grocery,Category_home,Store_Type_rural,Store_Type_urban,Festival_Name_christmas,Festival_Name_day,Festival_Name_dhanteras,Festival_Name_diwali,Festival_Name_dussehra,Festival_Name_eid,Festival_Name_fitr,Festival_Name_holi,Festival_Name_independence,Festival_Name_new,Festival_Name_ul,Festival_Name_year,Festival_Type_festive,Festival_Type_gifting,Festival_Type_mega,Festival_Type_season,Festival_Type_wedding,Product_Name_1012,Product_Name_1014,Product_Name_1015,Product_Name_1016,Product_Name_1019,Product_Name_activewear,Product_Name_ashirvad,Product_Name_computers,Product_Name_fabindia,Product_Name_fragrance,Product_Name_kitchenware,Product_Name_lakme,Product_Name_makeup,Product_Name_nestle,Product_Name_samsung,Product_Name_sony,Product_Name_spices,Product_Name_tata,Product_Name_tupperware,Product_Name_wear,Sub_Category_accessories,Sub_Category_activewear,Sub_Category_audio,Sub_Category_bedding,Sub_Category_beverages,Sub_Category_computers,Sub_Category_decor,Sub_Category_ethnic,Sub_Category_formalwear,Sub_Category_fragrance,Sub_Category_furniture,Sub_Category_grains,Sub_Category_haircare,Sub_Category_kitchenware,Sub_Category_makeup,Sub_Category_mobiles,Sub_Category_skincare,Sub_Category_spices,Sub_Category_wear,Sub_Category_winter,Brand_Name_allen,Brand_Name_amul,Brand_Name_ashirvad,Brand_Name_bose,Brand_Name_dove,Brand_Name_fabindia,Brand_Name_ikea,Brand_Name_lakme,Brand_Name_levis,Brand_Name_nestle,Brand_Name_nivea,Brand_Name_oreal,Brand_Name_philips,Brand_Name_samsung,Brand_Name_sleepwell,Brand_Name_solly,Brand_Name_sony,Brand_Name_tata,Brand_Name_tupperware,Brand_Name_zara,Brand_Tier_budget,Brand_Tier_mid,Brand_Tier_premium,Promotion_Type_bogo,Promotion_Type_discount,Promotion_Type_flash,Promotion_Type_loyalty,Promotion_Type_only,Promotion_Type_sale,Promotion_Type_seasonal
0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,1.000000,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
1,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.728911,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.00000,0.684609,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.707107,0.0,0.0,0.707107,0.0
2,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.609405,0.000000,0.0,0.0,0.0,0.525581,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.593625,0.000000,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
3,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.600282,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.602928,0.0,0.000000,0.525490,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.707107,0.0,0.0,0.707107,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.606904,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.525423,0.59632,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0

## Observation

The column-wise TF-IDF approach preserves the semantic role of each feature by vectorizing each column independently.

Compared to the combined-text TF-IDF approach:

- semantic mixing is reduced
- feature meaning is preserved more clearly
- interpretability is improved

This makes the representation more suitable for structured datasets in which different text columns represent different business concepts.

However, these fields are still short categorical labels rather than natural language text. Therefore, although column-wise TF-IDF is an improvement over combined-text TF-IDF, traditional categorical encoding methods such as one-hot encoding and frequency encoding remain more appropriate for this dataset.

## Prepare Tokens for Word2Vec

Word2Vec requires tokenized text input.

So we convert each cleaned text row into a list of words.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create an empty list to store TF-IDF DataFrames for each column
tfidf_features = []

# Apply TF-IDF separately to each text column
for col in text_columns:
    # Initialize TF-IDF vectorizer
    # max_features is limited to keep the feature space manageable
    vectorizer = TfidfVectorizer(max_features=20)

    # Convert the current column to string
    text_data = df[col].astype(str)

    # Fit and transform the column
    col_matrix = vectorizer.fit_transform(text_data)

    # Convert sparse matrix to DataFrame
    col_df = pd.DataFrame(
        col_matrix.toarray(),
        columns=[f"{col}_{word}" for word in vectorizer.get_feature_names_out()]
    )

    # Store the result
    tfidf_features.append(col_df)

# Combine all column-wise TF-IDF features into one DataFrame
tfidf_combined_df = pd.concat(tfidf_features, axis=1)

# Display shape and preview
print("Column-wise TF-IDF shape:", tfidf_combined_df.shape)
tfidf_combined_df.head()

Column-wise TF-IDF shape: (2000, 94)


,Category_beauty,Category_clothing,Category_electronics,Category_grocery,Category_home,Store_Type_rural,Store_Type_urban,Festival_Name_christmas,Festival_Name_day,Festival_Name_dhanteras,Festival_Name_diwali,Festival_Name_dussehra,Festival_Name_eid,Festival_Name_fitr,Festival_Name_holi,Festival_Name_independence,Festival_Name_new,Festival_Name_ul,Festival_Name_year,Festival_Type_festive,Festival_Type_gifting,Festival_Type_mega,Festival_Type_season,Festival_Type_wedding,Product_Name_1012,Product_Name_1014,Product_Name_1015,Product_Name_1016,Product_Name_1019,Product_Name_activewear,Product_Name_ashirvad,Product_Name_computers,Product_Name_fabindia,Product_Name_fragrance,Product_Name_kitchenware,Product_Name_lakme,Product_Name_makeup,Product_Name_nestle,Product_Name_samsung,Product_Name_sony,Product_Name_spices,Product_Name_tata,Product_Name_tupperware,Product_Name_wear,Sub_Category_accessories,Sub_Category_activewear,Sub_Category_audio,Sub_Category_bedding,Sub_Category_beverages,Sub_Category_computers,Sub_Category_decor,Sub_Category_ethnic,Sub_Category_formalwear,Sub_Category_fragrance,Sub_Category_furniture,Sub_Category_grains,Sub_Category_haircare,Sub_Category_kitchenware,Sub_Category_makeup,Sub_Category_mobiles,Sub_Category_skincare,Sub_Category_spices,Sub_Category_wear,Sub_Category_winter,Brand_Name_allen,Brand_Name_amul,Brand_Name_ashirvad,Brand_Name_bose,Brand_Name_dove,Brand_Name_fabindia,Brand_Name_ikea,Brand_Name_lakme,Brand_Name_levis,Brand_Name_nestle,Brand_Name_nivea,Brand_Name_oreal,Brand_Name_philips,Brand_Name_samsung,Brand_Name_sleepwell,Brand_Name_solly,Brand_Name_sony,Brand_Name_tata,Brand_Name_tupperware,Brand_Name_zara,Brand_Tier_budget,Brand_Tier_mid,Brand_Tier_premium,Promotion_Type_bogo,Promotion_Type_discount,Promotion_Type_flash,Promotion_Type_loyalty,Promotion_Type_only,Promotion_Type_sale,Promotion_Type_seasonal
0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,1.000000,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
1,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.728911,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.00000,0.684609,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.707107,0.0,0.0,0.707107,0.0
2,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.609405,0.000000,0.0,0.0,0.0,0.525581,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.593625,0.000000,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
3,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.600282,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.602928,0.0,0.000000,0.525490,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.707107,0.0,0.0,0.707107,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.606904,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.525423,0.59632,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0

In [12]:
# Tokenize cleaned text by simple whitespace splitting
df["tokens"] = df["clean_text"].apply(lambda x: x.split())

# Preview tokenized text
df[["clean_text", "tokens"]].head()

,clean_text,tokens
0,grocery urban nestle beverages 1002 beverages nestle premium,"[grocery, urban, nestle, beverages, 1002, beverages, nestle, premium]"
1,home rural tupperware decor 1014 decor tupperware budget flash sale,"[home, rural, tupperware, decor, 1014, decor, tupperware, budget, flash, sale]"
2,electronics urban sony computers 1015 computers sony mid,"[electronics, urban, sony, computers, 1015, computers, sony, mid]"
3,grocery urban nestle spices 1012 spices nestle premium flash sale,"[grocery, urban, nestle, spices, 1012, spices, nestle, premium, flash, sale]"
4,grocery urban tata spices 1016 spices tata premium,"[grocery, urban, tata, spices, 1016, spices, tata, premium]"


# 3. Word2Vec

## Concept

Word2Vec converts words into **dense numerical embeddings**.

Unlike BoW and TF-IDF, Word2Vec can capture some degree of **semantic similarity** between words.

It learns word representations by examining surrounding words in context.

Two common architectures are:
- **CBOW**: predicts a word from surrounding words
- **Skip-Gram**: predicts surrounding words from a given word

## Why We Are Using It

Word2Vec provides:
- dense low-dimensional vectors
- better semantic representation than frequency-based methods

## Limitation

Word2Vec is **context-independent**, meaning a word gets the same embedding everywhere.

For this dataset, Word2Vec is exploratory because the text fields are short business labels rather than full sentences.

In [13]:
# Train a simple Word2Vec model on tokenized text
# vector_size = embedding dimension
# window = context window size
# min_count = minimum frequency required for a word to be learned
# workers = number of CPU cores used
w2v_model = Word2Vec(
    sentences=df["tokens"],
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    sg=1  # sg=1 means Skip-Gram
)

print("Word2Vec vocabulary size:", len(w2v_model.wv.index_to_key))
print("Sample words:", w2v_model.wv.index_to_key[:10])

Word2Vec vocabulary size: 96
Sample words: ['urban', 'rural', 'premium', 'mid', 'budget', 'grocery', 'beauty', 'home', 'electronics', 'computers']


## Create Document-Level Word2Vec Vectors

Word2Vec learns vectors for individual words, but our machine learning model later needs **one vector per row**.

So we compute the **average of all word vectors in each row**.

### Formula

For a document with $n$ words:

$
\text{Document Vector} = \frac{1}{n}\sum_{i=1}^{n} \vec{w_i}
$

where:
- $\vec{w_i}$ = embedding of word $i$

In [14]:
# Function to average word vectors for one document
def average_word2vec(tokens, model, vector_size):
    """
    Return the average Word2Vec embedding for a list of tokens.
    If no token is found, return a zero vector.
    """
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]

    if len(valid_vectors) == 0:
        return np.zeros(vector_size)

    return np.mean(valid_vectors, axis=0)

# Apply function row by row
w2v_vectors = np.array([
    average_word2vec(tokens, w2v_model, 50)
    for tokens in df["tokens"]
])

# Convert to DataFrame
w2v_df = pd.DataFrame(w2v_vectors, columns=[f"w2v_{i}" for i in range(50)])

print("Word2Vec document matrix shape:", w2v_df.shape)
w2v_df.head()

Word2Vec document matrix shape: (2000, 50)


,w2v_0,w2v_1,w2v_2,w2v_3,w2v_4,w2v_5,w2v_6,w2v_7,w2v_8,w2v_9,w2v_10,w2v_11,w2v_12,w2v_13,w2v_14,w2v_15,w2v_16,w2v_17,w2v_18,w2v_19,w2v_20,w2v_21,w2v_22,w2v_23,w2v_24,w2v_25,w2v_26,w2v_27,w2v_28,w2v_29,w2v_30,w2v_31,w2v_32,w2v_33,w2v_34,w2v_35,w2v_36,w2v_37,w2v_38,w2v_39,w2v_40,w2v_41,w2v_42,w2v_43,w2v_44,w2v_45,w2v_46,w2v_47,w2v_48,w2v_49
0,-0.092853,-0.078781,0.060864,0.057939,-0.145222,-0.038916,0.234718,0.117449,-0.239555,-0.066501,0.099180,-0.486961,0.201814,0.217943,-0.061744,0.153571,0.020941,0.041446,-0.172649,-0.166352,0.179930,0.324708,0.472049,-0.191222,0.081422,0.286348,-0.020276,0.118110,-0.193459,-0.030070,-0.177700,-0.481767,0.113250,-0.099940,-0.183477,0.242776,0.037340,0.047396,-0.103322,0.171193,0.238214,-0.003670,-0.169366,0.056596,0.307203,0.110787,0.018722,-0.231596,0.158131,0.148000
1,-0.049718,0.114319,0.163936,0.089554,-0.152529,0.139797,0.194684,0.526750,-0.042830,-0.169253,0.020874,-0.239007,0.232380,0.268796,-0.214699,-0.022034,0.152720,0.147543,-0.221057,-0.116703,0.148645,0.073457,0.554717,0.066154,0.243456,0.237691,-0.176621,0.427574,-0.206159,0.272440,-0.029912,-0.316144,0.123420,-0.126635,-0.073913,-0.051376,0.040172,0.046355,0.011149,0.119348,0.205388,-0.127950,-0.147774,-0.053626,0.168481,-0.272325,0.280231,-0.438868,-0.040024,0.031182
2,0.009395,0.186711,-0.116326,0.156620,0.175861,-0.239864,0.126047,0.136162,-0.130366,-0.132628,0.005319,-0.303921,0.138961,0.216313,-0.297463,0.021461,0.188965,0.185597,-0.179694,-0.099145,0.242254,0.363535,0.295761,-0.124858,0.069699,-0.008700,-0.097466,0.293911,-0.119754,-0.036777,-0.302990,-0.436356,0.195697,-0.104784,-0.276510,-0.032093,0.301736,0.296987,-0.145000,0.382843,0.319254,-0.188566,-0.095436,0.094580,0.434392,-0.157939,-0.003357,0.005053,0.244950,0.208609
3,-0.068717,-0.054978,0.119850,0.099447,-0.184870,-0.033369,0.247362,0.115194,-0.242147,-0.060342,0.094890,-0.436076,0.217028,0.190267,-0.066548,0.112078,0.033448,0.119414,-0.191920,-0.161345,0.167535,0.301401,0.482220,-0.168005,0.104268,0.261429,-0.048881,0.149484,-0.183948,0.022975,-0.108778,-0.419149,0.107424,-0.133682,-0.183206,0.161866,-0.005574,0.069364,-0.037646,0.161575,0.251661,0.008420,-0.165129,0.026597,0.317120,0.072661,0.051147,-0.265374,0.153415,0.083529
4,-0.112688,-0.091370,0.139225,0.101014,-0.277112,-0.042811,0.325036,0.018364,-0.257448,-0.022257,0.153748,-0.505112,0.260903,0.168787,-0.023511,0.158331,-0.023104,0.096796,-0.223699,-0.199262,0.129675,0.345709,0.538197,-0.164134,0.106573,0.265167,-0.003282,0.083166,-0.222334,-0.009097,-0.073188,-0.464700,0.097350,-0.196498,-0.230481,0.167753,-0.045638,0.077580,-0.026506,0.134925,0.273397,0.080816,-0.219793,-0.023117,0.365533,0.151120,-0.015934,-0.214387,0.232822,0.072761


## Observation

Word2Vec learned a vocabulary of **96 unique tokens** and produced a **2000 × 50** dense document embedding matrix after averaging word vectors row by row.

The sample vocabulary includes meaningful business words such as:
- `urban`, `rural`
- `premium`, `mid`, `budget`
- `grocery`, `beauty`, `home`, `electronics`
- `computers`

This indicates that Word2Vec is capturing semantic relationships among store, product, and brand-related terms rather than only raw counts. However, because each row contains only short label-style text instead of full natural-language sentences, the learned semantic context is limited. As a result, Word2Vec provides a compact dense representation, but its benefit in this dataset is still exploratory rather than strongly language-driven.


# 4. BERT / Transformer Embeddings

## Concept

BERT-based embeddings produce **context-aware vector representations**.

Unlike Word2Vec, BERT can generate embeddings that reflect the context in which words appear.

For this notebook, we use a lightweight sentence-transformer model to generate one embedding per row.

## Why We Are Using It

Transformer embeddings provide:
- richer semantic representation
- context-aware text encoding
- strong performance in modern NLP tasks

## Limitation

They are more computationally expensive than BoW, TF-IDF, or Word2Vec.

In this dataset, BERT is exploratory because the text fields are short labels and not long natural-language sentences.

In [15]:
# Load a lightweight sentence transformer model
bert_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for each text row
bert_vectors = bert_model.encode(
    df["clean_text"].tolist(),
    show_progress_bar=True
)

# Convert embeddings to DataFrame
bert_df = pd.DataFrame(
    bert_vectors,
    columns=[f"bert_{i}" for i in range(bert_vectors.shape[1])]
)

print("BERT embedding matrix shape:", bert_df.shape)
bert_df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

BERT embedding matrix shape: (2000, 384)


,bert_0,bert_1,bert_2,bert_3,bert_4,bert_5,bert_6,bert_7,bert_8,bert_9,bert_10,bert_11,bert_12,bert_13,bert_14,bert_15,bert_16,bert_17,bert_18,bert_19,bert_20,bert_21,bert_22,bert_23,bert_24,bert_25,bert_26,bert_27,bert_28,bert_29,bert_30,bert_31,bert_32,bert_33,bert_34,bert_35,bert_36,bert_37,bert_38,bert_39,bert_40,bert_41,bert_42,bert_43,bert_44,bert_45,bert_46,bert_47,bert_48,bert_49,bert_50,bert_51,bert_52,bert_53,bert_54,bert_55,bert_56,bert_57,bert_58,bert_59,bert_60,bert_61,bert_62,bert_63,bert_64,bert_65,bert_66,bert_67,bert_68,bert_69,bert_70,bert_71,bert_72,bert_73,bert_74,bert_75,bert_76,bert_77,bert_78,bert_79,bert_80,bert_81,bert_82,bert_83,bert_84,bert_85,bert_86,bert_87,bert_88,bert_89,bert_90,bert_91,bert_92,bert_93,bert_94,bert_95,bert_96,bert_97,bert_98,bert_99,bert_100,bert_101,bert_102,bert_103,bert_104,bert_105,bert_106,bert_107,bert_108,bert_109,bert_110,bert_111,bert_112,bert_113,bert_114,bert_115,bert_116,bert_117,bert_118,bert_119,bert_120,bert_121,bert_122,bert_123,bert_124,bert_125,bert_126,bert_127,bert_128,bert_129,bert_130,bert_131,bert_132,bert_133,bert_134,bert_135,bert_136,bert_137,bert_138,bert_139,bert_140,bert_141,bert_142,bert_143,bert_144,bert_145,bert_146,bert_147,bert_148,bert_149,bert_150,bert_151,bert_152,bert_153,bert_154,bert_155,bert_156,bert_157,bert_158,bert_159,bert_160,bert_161,bert_162,bert_163,bert_164,bert_165,bert_166,bert_167,bert_168,bert_169,bert_170,bert_171,bert_172,bert_173,bert_174,bert_175,bert_176,bert_177,bert_178,bert_179,bert_180,bert_181,bert_182,bert_183,bert_184,bert_185,bert_186,bert_187,bert_188,bert_189,bert_190,bert_191,bert_192,bert_193,bert_194,bert_195,bert_196,bert_197,bert_198,bert_199,bert_200,bert_201,bert_202,bert_203,bert_204,bert_205,bert_206,bert_207,bert_208,bert_209,bert_210,bert_211,bert_212,bert_213,bert_214,bert_215,bert_216,bert_217,bert_218,bert_219,bert_220,bert_221,bert_222,bert_223,bert_224,bert_225,bert_226,bert_227,bert_228,bert_229,bert_230,bert_231,bert_232,bert_233,bert_234,bert_235,bert_236,bert_237,bert_238,bert_239,bert_240,bert_241,bert_242,bert_243,bert_244,bert_245,bert_246,bert_247,bert_248,bert_249,bert_250,bert_251,bert_252,bert_253,bert_254,bert_255,bert_256,bert_257,bert_258,bert_259,bert_260,bert_261,bert_262,bert_263,bert_264,bert_265,bert_266,bert_267,bert_268,bert_269,bert_270,bert_271,bert_272,bert_273,bert_274,bert_275,bert_276,bert_277,bert_278,bert_279,bert_280,bert_281,bert_282,bert_283,bert_284,bert_285,bert_286,bert_287,bert_288,bert_289,bert_290,bert_291,bert_292,bert_293,bert_294,bert_295,bert_296,bert_297,bert_298,bert_299,bert_300,bert_301,bert_302,bert_303,bert_304,bert_305,bert_306,bert_307,bert_308,bert_309,bert_310,bert_311,bert_312,bert_313,bert_314,bert_315,bert_316,bert_317,bert_318,bert_319,bert_320,bert_321,bert_322,bert_323,bert_324,bert_325,bert_326,bert_327,bert_328,bert_329,bert_330,bert_331,bert_332,bert_333,bert_334,bert_335,bert_336,bert_337,bert_338,bert_339,bert_340,bert_341,bert_342,bert_343,bert_344,bert_345,bert_346,bert_347,bert_348,bert_349,bert_350,bert_351,bert_352,bert_353,bert_354,bert_355,bert_356,bert_357,bert_358,bert_359,bert_360,bert_361,bert_362,bert_363,bert_364,bert_365,bert_366,bert_367,bert_368,bert_369,bert_370,bert_371,bert_372,bert_373,bert_374,bert_375,bert_376,bert_377,bert_378,bert_379,bert_380,bert_381,bert_382,bert_383
0,0.019519,0.035894,0.006501,0.005474,0.032532,0.009000,-0.032032,0.010910,-0.001356,-0.006492,-0.007242,-0.069013,-0.059507,-0.016442,0.036766,-0.038140,0.009790,-0.029988,0.021914,-0.093959,-0.010006,-0.072440,0.035189,0.006244,0.033071,0.092676,0.016329,0.058809,-0.013505,-0.024259,0.062140,0.045098,0.073857,-0.013253,0.001952,-0.027706,0.068708,-0.094382,0.014232,-0.036728,0.103842,-0.042366,-0.029061,-0.011378,-0.021398,-0.001483,-0.095435,0.106851,0.022592,0.022983,-0.091872,-0.011558,0.020245,-0.039998,0.005176,0.003343,-0.025824,-0.025057,0.040420,0.045489,0.015363,-0.010060,-0.088033,0.011045,-0.050195,0.009212,-0.081680,0.058867,0.0028

## Observation

BERT produced a **2000 × 384** embedding matrix, which is the richest and highest-dimensional representation among the four methods tested.

This method generates contextual sentence-level embeddings, so each row is represented by a dense vector rather than a sparse count table. In theory, this is more expressive than BoW, TF-IDF, or Word2Vec.

However, in this dataset the input text is still composed mostly of short business labels such as category names, brand names, product labels, promotion types, and identifiers. Because the rows are not long natural-language descriptions, BERT's contextual advantage is present but likely underutilized. Therefore, BERT provides the most advanced representation technically, but its practical gain for this structured retail dataset may be limited.


# Compare the Four Vectorization Outputs

This section compares the dimensionality of the four vectorization methods.

In [16]:
# Compare the shape of all vector outputs
comparison_df = pd.DataFrame({
    "Method": ["Bag of Words", "TF-IDF", "Word2Vec", "BERT"],
    "Rows": [
        bow_df.shape[0],
        tfidf_df.shape[0],
        w2v_df.shape[0],
        bert_df.shape[0]
    ],
    "Columns / Features": [
        bow_df.shape[1],
        tfidf_df.shape[1],
        w2v_df.shape[1],
        bert_df.shape[1]
    ]
})

comparison_df

,Method,Rows,Columns / Features
0,Bag of Words,2000,95
1,TF-IDF,2000,95
2,Word2Vec,2000,50
3,BERT,2000,384


# Summary and Interpretation

The four vectorization methods were implemented successfully on the text-like fields of the updated retail dataset.

## Summary of Methods

- **Bag of Words (BoW)**  
  Produced a **2000 × 95** sparse matrix based on token counts.

- **TF-IDF**  
  Produced a **2000 × 95** weighted matrix that emphasizes more distinctive terms.

- **Word2Vec**  
  Learned a **96-token vocabulary** and generated a **2000 × 50** dense embedding matrix.

- **BERT / Transformer Embeddings**  
  Produced a **2000 × 384** contextual embedding matrix.

## Overall Interpretation

The results confirm that these methods can be applied successfully to the selected text-like fields. However, the underlying text in this dataset is mostly composed of:

- short categorical labels
- brand and product names
- promotion labels
- numeric identifiers

rather than long natural-language text.

As a result:

- **BoW** and **TF-IDF** mainly capture token presence and relative importance
- **Word2Vec** captures limited semantic similarity among business terms
- **BERT** provides the richest representation, but its contextual power is not fully utilized in short label-based records

## Practical Conclusion

These vectorization methods satisfy the project requirement and are useful for comparative analysis. However, for this retail dataset, they should be treated as **secondary / exploratory text representations** rather than the primary feature engineering strategy.

More suitable methods for the full forecasting pipeline are likely to be:

- categorical encoding
- date and calendar feature extraction
- lag features
- rolling statistics
- price-promotion interaction features
- store/product aggregation features


## Optional Step: Save Vectorized Outputs

If needed, the generated feature matrices can be saved for use in later machine learning experiments.

In [17]:
# Save vectorized outputs as CSV files for later use
bow_df.to_csv("bow_features.csv", index=False)
tfidf_df.to_csv("tfidf_features.csv", index=False)
w2v_df.to_csv("word2vec_features.csv", index=False)
bert_df.to_csv("bert_features.csv", index=False)

print("Feature files saved successfully.")

Feature files saved successfully.


## Final Conclusion

The four vectorization methods were applied successfully to the text-like fields of the updated retail dataset.

From the actual outputs:
- **BoW** and **TF-IDF** each produced **95 features**
- **Word2Vec** produced **50 dense features**
- **BERT** produced **384 dense contextual features**

These results show that the text-like fields can be transformed into machine-readable vectors. However, since the dataset is primarily structured and categorical rather than text-heavy, these methods are best viewed as exploratory approaches for this project.

For the broader retail demand forecasting task, more suitable feature engineering methods will include categorical encoding, calendar-based features, lag variables, rolling statistics, interaction variables, and aggregation-based features.


# Summary and Interpretation

The text-like fields of the updated retail dataset were explored using four vectorization methods:

- Bag of Words (BoW)
- TF-IDF
- Word2Vec
- BERT / Transformer Embeddings

In addition, an improved **column-wise TF-IDF** approach was tested to preserve the semantic meaning of individual text-like columns.

## Key Findings

- BoW and TF-IDF on combined text provide baseline vector representations.
- However, combining all text-like fields into one document mixes different business meanings such as product, brand, promotion, and store type.
- Column-wise TF-IDF improves interpretability by preserving feature-specific context.
- Word2Vec and BERT generate dense embeddings, but their value is limited because the dataset does not contain long natural-language text.

## Final Interpretation

Although these vectorization methods were successfully applied, the dataset is fundamentally a **structured categorical retail dataset**, not a text-heavy NLP dataset.

Therefore, the most appropriate techniques for this dataset are still:

- one-hot encoding
- frequency encoding
- target encoding
- business-oriented feature engineering

The vectorization methods are useful as exploratory techniques, but they are not the primary feature engineering strategy for this problem.

## Comparison: Combined TF-IDF vs Column-wise TF-IDF

The two TF-IDF strategies differ in how they treat the text-like columns.

| Approach | Description | Strength | Limitation |
|----------|-------------|----------|------------|
| Combined TF-IDF | All text columns merged into one document per row | Simple baseline | mixes unrelated semantics |
| Column-wise TF-IDF | Each text column vectorized separately | preserves feature meaning better | still less suitable than categorical encoding for this dataset |